<a href="https://colab.research.google.com/github/AmarSai01/nvidia-gpu-ml-jax-labs/blob/main/rapids-data-analytics/NYC_Taxi_Data_Analytics_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Wed Oct 28 21:27:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!for m in 01 02 03; do wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-$m.parquet; done
!ls -lh *.parquet

-rw-r--r-- 1 root root 48M Mar 21  2024 yellow_tripdata_2024-01.parquet
-rw-r--r-- 1 root root 49M Apr 19  2024 yellow_tripdata_2024-02.parquet
-rw-r--r-- 1 root root 58M May 22  2024 yellow_tripdata_2024-03.parquet


In [3]:
import pandas as pd, time, glob
files = sorted(glob.glob("yellow_tripdata_2024-*.parquet"))

t = time.time()
df = pd.concat([pd.read_parquet(f) for f in files])
df["hour"] = df.tpep_pickup_datetime.dt.hour
r1 = df.groupby("hour")["fare_amount"].mean()
r2 = df.groupby("PULocationID")["tip_amount"].agg(["mean", "count"]).sort_values("count", ascending=False).head(10)
cpu_time = time.time() - t
print(df.shape, f"CPU time: {cpu_time:.1f}s")
r2

(9554778, 20) CPU time: 4.9s


,mean,count
PULocationID,,
161,3.101891,453826
237,2.595715,439139
132,9.004939,429746
236,2.633889,416509
162,3.050865,336460
230,3.316136,331570
186,3.102850,319706
142,2.764969,316314
138,8.846030,284362


In [4]:
import cudf
t = time.time()
gdf = cudf.concat([cudf.read_parquet(f) for f in files])
gdf["hour"] = gdf.tpep_pickup_datetime.dt.hour
g1 = gdf.groupby("hour")["fare_amount"].mean()
g2 = gdf.groupby("PULocationID")["tip_amount"].agg(["mean", "count"]).sort_values("count", ascending=False).head(10)
gpu_time = time.time() - t
print(f"GPU time: {gpu_time:.1f}s  ->  {cpu_time/gpu_time:.1f}x faster")
g2

GPU time: 1.7s  ->  2.9x faster


,mean,count
PULocationID,,
161,3.101891,453826
237,2.595715,439139
132,9.004939,429746
236,2.633889,416509
162,3.050865,336460
230,3.316136,331570
186,3.102850,319706
142,2.764969,316314
138,8.846030,284362


In [5]:
df.head(2_000_000).to_csv("sample.csv", index=False)
df.head(2_000_000).to_parquet("sample.parquet")
!ls -lh sample.*
t = time.time(); pd.read_csv("sample.csv"); print(f"CSV read: {time.time()-t:.1f}s")
t = time.time(); pd.read_parquet("sample.parquet"); print(f"Parquet read: {time.time()-t:.1f}s")

-rw-r--r-- 1 root root 207M Oct 28 21:28 sample.csv
-rw-r--r-- 1 root root  48M Oct 28 21:28 sample.parquet
CSV read: 4.1s
Parquet read: 0.6s


In [2]:
%load_ext cudf.pandas
import pandas as pd
df = pd.read_parquet("yellow_tripdata_2024-01.parquet")

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [3]:
%%cudf.pandas.profile
df["hour"] = df.tpep_pickup_datetime.dt.hour
df.groupby("hour")["fare_amount"].mean()

,fare_amount
hour,
0,19.202658
1,17.527296
2,16.482882
3,18.150135
4,22.518645
5,26.619918
6,21.650399
7,18.539180
8,17.654683


                                                                                                                  
                                            Total time elapsed: 1.480 seconds                                     
                                          7 GPU function calls in 0.162 seconds                                   
                                          2 CPU function calls in 0.014 seconds                                   
                                                                                                                  
                                                          Stats                                                   
                                                                                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                     ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Series                       │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__        │ 1          │ 0.004       │ 0.004       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.groupby            │ 1          │ 0.068       │ 0.068       │ 0          │ 0.000       │ 0.000       │
│ DataFrameGroupBy.__getitem__ │ 1          │ 0.004       │ 0.004       │ 0          │ 0.000       │ 0.000       │
│ GroupBy.mean                 │ 1          │ 0.079       │ 0.079       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__              │ 1          │ 0.006       │ 0.006       │ 0          │ 0.000       │ 0.000       │
│ Series.to_frame              │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ DataFrame._repr_html_        │ 0          │ 0.000       │ 0.000       │ 1          │ 0.012       │ 0.012       │
│ NDFrame._repr_latex_         │ 0          │ 0.000       │ 0.000       │ 1          │ 0.001       │ 0.001       │
└──────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- DataFrame._repr_html_
- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=387686;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.